In [0]:
with t1 as (
  SELECT PATIENT_ID, patient_state
    FROM com_edp_prd.com_raw.kom_patient_geography
    WHERE VALID_TO_DATE > CURRENT_DATE()
)
select patient_id, count(distinct patient_state)
from t1
group by 1 order by 2 desc

In [0]:
with all_most_seen_hcp as (select distinct most_seen_hcp1_3yr_ranked as npi
from com_edp_prd.cmpa_insights_internal_schema.patient360
union
select distinct most_seen_hcp2_3yr_ranked as npi
from com_edp_prd.cmpa_insights_internal_schema.patient360
union
select distinct most_seen_hcp3_3yr_ranked as npi
from com_edp_prd.cmpa_insights_internal_schema.patient360
union
select distinct most_seen_hcp4_3yr_ranked as npi
from com_edp_prd.cmpa_insights_internal_schema.patient360
union
select distinct most_seen_hcp5_3yr_ranked as npi
from com_edp_prd.cmpa_insights_internal_schema.patient360)
select distinct hcp_npi
from com_edp_prd.cmpa_insights_internal_schema.reference_file_0109
where hcp_npi in (select * from all_most_seen_hcp)

In [0]:
select * from cmpa_insights_internal_schema.tableau_wide_view_pt360

In [0]:
select a.*, concat(b.FIRST_NAME, ' ', b.LAST_NAME)
from (select first_dx_hcp_5yr, first_dx_hcp_name_5yr from cmpa_insights_internal_schema.tableau_wide_view_pt360
where first_dx_hcp_name_5yr is null and first_dx_hcp_5yr is not null) as a
left join com_edp_prd.com_raw.kom_providers as b on a.first_dx_hcp_5yr = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'

In [0]:
WITH base_npis AS (
    select distinct hco_npi_old as npi
    from cmpa_insights_internal_schema.reference_file
),

hco_engaged AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_edp_prd.com_raw.vcrm_call2__v a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
    WHERE a.call_date__v BETWEEN '2025-07-01' AND '2025-12-31'
),

hco_profiled AS (
    SELECT DISTINCT TRY_CAST(b.npi__v AS STRING) AS npi
    FROM com_intgr.survey_target a
    JOIN com_intgr.customer b
        ON a.account__v = b.id
)

SELECT
    b.npi,
    CASE WHEN e.npi IS NOT NULL THEN 1 ELSE 0 END AS is_engaged,
    CASE WHEN p.npi IS NOT NULL THEN 1 ELSE 0 END AS is_profiled
FROM base_npis b
LEFT JOIN hco_engaged e
    ON b.npi = e.npi
LEFT JOIN hco_profiled p
    ON b.npi = p.npi
ORDER BY b.npi;


In [0]:
with t1 as (select distinct account__v, account_display_name__v
from com_intgr.survey_target
where id in ('VC3000000006003', 'VC3000000008003', 'VC3000000009001', 'VC3000000009002', 'VC300000000A001', 'VC300000000A006', 'VC300000000D001', 'VC300000000E001', 'VC300000000E002', 'VC300000000F001', 'VC300000000F002', 'VC300000000H007', 'VC300000000I001', 'VC300000000J002', 'VC300000000K001', 'VC300000000L002', 'VC300000000N001', 'VC300000000O001', 'VC300000000P010', 'VC300000000R001', 'VC300000000S001'
))
select a.*, b.npi__v, c.hco_npi, c.hco_name
from t1 as a
left join com_intgr.customer as b on a.account__v = b.id
left join cmpa_insights_internal_schema.reference_file_0109 as c on b.npi__v = c.hcp_npi